om namah shivaay!

In [17]:
import pandas as pd
import os
from utils import preprocess_text
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix
import joblib

In [18]:
path=os.path.join('data','amazon_review.csv')
try:
    if path:
        data=pd.read_csv(path)
        print("file loaded successfully!")
except Exception as e:
    print("file not found!",e)

file loaded successfully!


In [19]:
data.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote
0,A3SBTW3WS4IQSN,B007WTAJTO,NaN,"[0, 0]",No issues.,4.0,Four Stars,1406073600,2014-07-23,138,0,0
1,A18K1ODH1I2MVB,B007WTAJTO,0mie,"[0, 0]","Purchased this for my device, it worked as adv...",5.0,MOAR SPACE!!!,1382659200,2013-10-25,409,0,0
2,A2FII3I2MBMUIA,B007WTAJTO,1K3,"[0, 0]",it works as expected. I should have sprung for...,4.0,nothing to really say....,1356220800,2012-12-23,715,0,0
3,A3H99DFEG68SR,B007WTAJTO,1m2,"[0, 0]",This think has worked out great.Had a diff. br...,5.0,Great buy at this price!!! *** UPDATE,1384992000,2013-11-21,382,0,0
4,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"[0, 0]","Bought it with Retail Packaging, arrived legit...",5.0,best deal around,1373673600,2013-07-13,513,0,0


In [20]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4915 entries, 0 to 4914
Data columns (total 12 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   reviewerID      4915 non-null   object 
 1   asin            4915 non-null   object 
 2   reviewerName    4914 non-null   object 
 3   helpful         4915 non-null   object 
 4   reviewText      4914 non-null   object 
 5   overall         4915 non-null   float64
 6   summary         4915 non-null   object 
 7   unixReviewTime  4915 non-null   int64  
 8   reviewTime      4915 non-null   object 
 9   day_diff        4915 non-null   int64  
 10  helpful_yes     4915 non-null   int64  
 11  total_vote      4915 non-null   int64  
dtypes: float64(1), int64(4), object(7)
memory usage: 460.9+ KB


In [21]:
data.isnull().sum()

reviewerID        0
asin              0
reviewerName      1
helpful           0
reviewText        1
overall           0
summary           0
unixReviewTime    0
reviewTime        0
day_diff          0
helpful_yes       0
total_vote        0
dtype: int64

In [22]:
data['overall']
positive=data[data['overall']>3]
negative=data[data['overall']<3]
neutral=data[data['overall']==3]
print(f"Positive:{len(positive)} \nNegative:{len(negative)}\nNeutral:{len(neutral)}")

Positive:4449 
Negative:324
Neutral:142


In [23]:
data['target']=data['overall'].apply(lambda x: 2 if x>3 else (1 if x==3 else 0))
data.head()

,reviewerID,asin,reviewerName,helpful,reviewText,overall,summary,unixReviewTime,reviewTime,day_diff,helpful_yes,total_vote,target
0,A3SBTW3WS4IQSN,B007WTAJTO,NaN,"[0, 0]",No issues.,4.0,Four Stars,1406073600,2014-07-23,138,0,0,2
1,A18K1ODH1I2MVB,B007WTAJTO,0mie,"[0, 0]","Purchased this for my device, it worked as adv...",5.0,MOAR SPACE!!!,1382659200,2013-10-25,409,0,0,2
2,A2FII3I2MBMUIA,B007WTAJTO,1K3,"[0, 0]",it works as expected. I should have sprung for...,4.0,nothing to really say....,1356220800,2012-12-23,715,0,0,2
3,A3H99DFEG68SR,B007WTAJTO,1m2,"[0, 0]",This think has worked out great.Had a diff. br...,5.0,Great buy at this price!!! *** UPDATE,1384992000,2013-11-21,382,0,0,2
4,A375ZM4U047O79,B007WTAJTO,2&amp;1/2Men,"[0, 0]","Bought it with Retail Packaging, arrived legit...",5.0,best deal around,1373673600,2013-07-13,513,0,0,2


In [24]:
data['reviews']=data['reviewText'].str.cat(data['summary'],sep=' ')

In [25]:
data.dropna(subset=['reviews'],inplace=True)
data['reviews'].isnull().sum()
data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4914 entries, 0 to 4914
Data columns (total 14 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   reviewerID      4914 non-null   object 
 1   asin            4914 non-null   object 
 2   reviewerName    4913 non-null   object 
 3   helpful         4914 non-null   object 
 4   reviewText      4914 non-null   object 
 5   overall         4914 non-null   float64
 6   summary         4914 non-null   object 
 7   unixReviewTime  4914 non-null   int64  
 8   reviewTime      4914 non-null   object 
 9   day_diff        4914 non-null   int64  
 10  helpful_yes     4914 non-null   int64  
 11  total_vote      4914 non-null   int64  
 12  target          4914 non-null   int64  
 13  reviews         4914 non-null   object 
dtypes: float64(1), int64(5), object(8)
memory usage: 575.9+ KB


In [26]:
data['reviews']=data['reviews'].apply(preprocess_text)


In [27]:
positive_data=data[data['target']==2]
negative_data=data[data['target']==0]
neutral_data=data[data['target']==1]
pos_downsampled=positive_data.sample(n=len(negative_data),random_state=42)
neu_upsampled=neutral_data.sample(n=len(negative_data),replace=True,random_state=42)
balanced_data=pd.concat([pos_downsampled,neu_upsampled,negative_data])
balanced_data.shape

(972, 14)

In [28]:
X=balanced_data['reviews']
y=balanced_data['target']
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [29]:
pipeline=[
    ('tfidf',TfidfVectorizer(analyzer='word',ngram_range=(1,2),preprocessor=preprocess_text,sublinear_tf=True)),
    ('classifier',MultinomialNB())
]
pipe=Pipeline(pipeline)
model=pipe.fit(X_train,y_train)
y_pred=model.predict(X_test)





In [30]:
accuracy=accuracy_score(y_test,y_pred)
print(f"Model Accuracy:{accuracy*100:.2f}%")

class_report_result=classification_report(y_test,y_pred)
print("Classification Report:\n",class_report_result)

conf_matrix=confusion_matrix(y_test,y_pred)
print("Confusion Matrix:\n",conf_matrix)

Model Accuracy:81.54%
Classification Report:
               precision    recall  f1-score   support

           0       0.87      0.82      0.84        65
           1       0.76      0.84      0.80        70
           2       0.84      0.78      0.81        60

    accuracy                           0.82       195
   macro avg       0.82      0.81      0.82       195
weighted avg       0.82      0.82      0.82       195

Confusion Matrix:
 [[53 10  2]
 [ 4 59  7]
 [ 4  9 47]]


In [31]:
test_samples = [
    "I hate this thing, it broke in two days.", # Clear Negative
    "Actually, I was surprised by how good it was.", # Subtle Positive
    "Do not buy this. It is a total waste of money. 😡" # Emoji + Negation
]

predictions = model.predict(test_samples)
print(predictions)

[0 1 0]


In [32]:
joblib.dump(model,os.path.join('model','sentiment_model.pkl'))

['model\\sentiment_model.pkl']